# 🧮 Árvore Geradora Mínima (AGM / Minimum Spanning Tree)

Em um grafo não dirigido e **ponderado** $G=(V,E,w)$, uma **árvore geradora mínima (AGM)** é uma árvore geradora cuja soma dos pesos das arestas é mínima entre todas as árvores geradoras de $G$.

- Peso de uma aresta: $w(e)$.
- Peso do grafo (ou da árvore): $w(H) = um_{en E(H)} w(e)$.

Aplicação: planejar rotas (ex.: companhias aéreas, rede elétrica, fibra óptica) minimizando custo total.

## ⚙️ Algoritmos Clássicos
- **Kruskal**: adiciona arestas em ordem crescente de peso, evitando ciclos via estrutura **Union-Find** (Disjoint Set Union — DSU).
- **Prim**: expande uma árvore a partir de um vértice inicial, escolhendo sempre a aresta de menor peso que conecta a árvore a um novo vértice (usa **fila de prioridade**).

In [ ]:
import heapq
import networkx as nx
import matplotlib.pyplot as plt
from typing import Any, Dict, List, Tuple, Iterable

class UnionFind:
    def __init__(self, elements: Iterable[Any]):
        self.parent = {x: x for x in elements}
        self.rank = {x: 0 for x in elements}
    def find(self, x):
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])
        return self.parent[x]
    def union(self, x, y) -> bool:
        rx, ry = self.find(x), self.find(y)
        if rx == ry:
            return False
        if self.rank[rx] < self.rank[ry]:
            self.parent[rx] = ry
        elif self.rank[rx] > self.rank[ry]:
            self.parent[ry] = rx
        else:
            self.parent[ry] = rx
            self.rank[rx] += 1
        return True

def kruskal_mst(G: nx.Graph) -> Tuple[List[Tuple[Any,Any,float]], float]:
    if G.is_directed():
        raise ValueError('Kruskal requer grafo não dirigido')
    edges = []
    for u, v, data in G.edges(data=True):
        w = data.get('weight', 1.0)
        edges.append((w, u, v))
    edges.sort()
    uf = UnionFind(G.nodes())
    mst_edges: List[Tuple[Any,Any,float]] = []
    total = 0.0
    for w, u, v in edges:
        if uf.union(u, v):
            mst_edges.append((u, v, w))
            total += w
            if len(mst_edges) == G.number_of_nodes() - 1:
                break
    if len(mst_edges) != G.number_of_nodes() - 1:
        raise ValueError('Grafo não é conexo — MST não existe')
    return mst_edges, total

def prim_mst(G: nx.Graph, start=None) -> Tuple[List[Tuple[Any,Any,float]], float]:
    if G.is_directed():
        raise ValueError('Prim requer grafo não dirigido')
    if start is None:
        start = next(iter(G.nodes()))
    visited = set([start])
    heap = []
    for v, data in G[start].items():
        heapq.heappush(heap, (data.get('weight', 1.0), start, v))
    mst_edges: List[Tuple[Any,Any,float]] = []
    total = 0.0
    while heap and len(visited) < G.number_of_nodes():
        w, u, v = heapq.heappop(heap)
        if v in visited:
            continue
        visited.add(v)
        mst_edges.append((u, v, w))
        total += w
        for x, data in G[v].items():
            if x not in visited:
                heapq.heappush(heap, (data.get('weight', 1.0), v, x))
    if len(mst_edges) != G.number_of_nodes() - 1:
        raise ValueError('Grafo não é conexo — MST não existe')
    return mst_edges, total

def draw_weighted_graph_with_mst(G: nx.Graph, mst_edges: List[Tuple[Any,Any,float]]):
    pos = nx.spring_layout(G, seed=3)
    fig, ax = plt.subplots(figsize=(12, 5), constrained_layout=True, constrained_layout=True)
    nx.draw(G, pos, with_labels=True, node_color='lightblue', node_size=900, font_size=12, ax=ax)
    labels = {(u,v): f'{data.get(
,1.0):.1f}' for u,v,data in G.edges(data=True)}
    nx.draw_networkx_edge_labels(G, pos, edge_labels=labels, font_color='darkgreen', ax=ax)
    # destacar MST
    mst_set = {(min(u,v), max(u,v)) for (u,v,_) in mst_edges}
    colored = []
    for u, v in G.edges():
        key = (min(u,v), max(u,v))
        colored.append('crimson' if key in mst_set else 'gray')
    nx.draw(G, pos, with_labels=True, node_color='lightblue', node_size=900, font_size=12, edge_color=colored, width=3, ax=ax)
    ax.set_title('Grafo ponderado com MST (arestas em vermelho)')
    plt.show()

## 🧪 Exemplo 1: MST com Kruskal e Prim

In [ ]:
G = nx.Graph()
G.add_weighted_edges_from([
    ('A','B',4), ('A','H',8), ('B','H',11), ('B','C',8), ('C','I',2), ('C','F',4),
    ('C','D',7), ('D','E',9), ('D','F',14), ('E','F',10), ('H','I',7), ('H','G',1),
    ('G','I',6), ('G','F',2)
])
mst_k, total_k = kruskal_mst(G)
mst_p, total_p = prim_mst(G, start='A')
print('Kruskal MST peso total:', total_k, '| arestas:', mst_k)
print('Prim    MST peso total:', total_p, '| arestas:', mst_p)

# Verificação com NetworkX
Tnx = nx.minimum_spanning_tree(G, weight='weight')
total_nx = sum(data['weight'] for _,_,data in Tnx.edges(data=True))
print('NetworkX MST peso total:', total_nx, '| arestas:', list(Tnx.edges(data='weight')))

draw_weighted_graph_with_mst(G, mst_k)

## ⏱️ Complexidade
- Kruskal: ordena arestas O(|E| log |E|), operações de união/busca quase O(1) amortizado → O(|E| log |E|).
- Prim: com fila de prioridade binária (heap), O(|E| log |V|).
- Ambos assumem grafo não dirigido e conexo (para existir AGM).

## 🧩 Exercícios
1. Gere um grafo ponderado aleatório e compare os MSTs de Kruskal e Prim (peso total e arestas).
2. Altere o peso de uma única aresta e observe se o MST muda.
3. Implemente Prim usando um dicionário de chaves (sem heap) e compare desempenho.
4. Verifique que todo MST tem exatamente |V|-1 arestas e não contém ciclos.
5. Modele o problema de rotas da companhia aérea com suas próprias cidades e distâncias, e obtenha a AGM.